# 1) Challenges in production deployment

In [1]:
import random, json

In [2]:
# Simulated LLM responses for the same task
def fake_llm_response(user_query):
    responses = [
        '{"action": "approve_loan", "amount": 50000, "confidence": 0.91}',
        '{"action": "reject_loan", "reason": "Low credit score", "confidence": 0.88}',
        'Approve the loan immediately',   # unpredictable free text
        '{"action": "approve_loan", "amount": "fifty thousand"}',  # wrong data type
        '{"decision": "approve"}',       # missing required keys
        'ERROR: I am not sure what to do'
    ]
    return random.choice(responses)

# Expected output schema
required_keys = ["action", "confidence"]

valid_actions = ["approve_loan", "reject_loan", "manual_review"]


def validate_llm_output(llm_output):
    """
    Validates whether the LLM output is reliable and usable.
    """

    try:
        data = json.loads(llm_output)
    except json.JSONDecodeError:
        return { "valid": False, "reason": "Output is not valid JSON", "safe_action": "manual_review" }

    # Check required keys
    for key in required_keys:
        if key not in data:
            return { "valid": False, "reason": f"Missing required key: {key}", "safe_action": "manual_review"}

    # Check valid action
    if data["action"] not in valid_actions:
        return { "valid": False, "reason": "Invalid action returned by LLM", "safe_action": "manual_review"}

    # Check confidence
    if not isinstance(data["confidence"], float):
        return { "valid": False, "reason": "Confidence must be a float value", "safe_action": "manual_review" }

    if data["confidence"] < 0.85:
        return { "valid": False, "reason": "Low confidence decision", "safe_action": "manual_review" }

    return { "valid": True, "reason": "LLM output is valid", "safe_action": data["action"], "data": data}

In [3]:
# Run simulation

user_query = "Evaluate this loan application"
llm_output = fake_llm_response(user_query)

print("LLM Output:", llm_output)

result = validate_llm_output(llm_output)

print("Validation Result:", result["reason"])
print("Final Action:", result["safe_action"])

LLM Output: ERROR: I am not sure what to do
Validation Result: Output is not valid JSON
Final Action: manual_review


# 2) Validate outputs through Deterministic Business Rules

In [6]:
import random

# ----------------------------
# Simulated LLM output
# ----------------------------
def fake_llm_decision(customer):
    decisions = [
        {"decision": "approve_loan", "loan_amount": 500000},
        {"decision": "approve_loan", "loan_amount": 2000000},
        {"decision": "reject_loan", "loan_amount": 0},
        {"decision": "approve_loan", "loan_amount": 300000},
    ]
    return random.choice(decisions)


# ----------------------------
# Deterministic business rules
# ----------------------------
def validate_with_business_rules(customer, llm_output):
    rules_failed = []

    if customer["credit_score"] < 700 and llm_output["decision"] == "approve_loan":
        rules_failed.append("Credit score is below 700")

    if llm_output["loan_amount"] > customer["annual_income"] * 3:
        rules_failed.append("Loan amount exceeds 3x annual income")

    if customer["existing_loans"] > 2 and llm_output["decision"] == "approve_loan":
        rules_failed.append("Customer already has too many loans")

    if rules_failed:
        return { "final_decision": "manual_review", "reason": rules_failed }

    return { "final_decision": llm_output["decision"], "reason": "All business rules passed" }


# ----------------------------
# Sample customer data
# ----------------------------
customer = { "customer_id": "C101", "name": "Arun", "credit_score": 650, "annual_income": 600000, "existing_loans": 1 }


# ----------------------------
# Run simulation
# ----------------------------
llm_output = fake_llm_decision(customer)

print("Customer:", customer)
print("LLM Output:", llm_output)

validated_result = validate_with_business_rules(customer, llm_output)

print("Final Decision:", validated_result["final_decision"])
print("Reason:", validated_result["reason"])

Customer: {'customer_id': 'C101', 'name': 'Arun', 'credit_score': 650, 'annual_income': 600000, 'existing_loans': 1}
LLM Output: {'decision': 'approve_loan', 'loan_amount': 2000000}
Final Decision: manual_review
Reason: ['Credit score is below 700', 'Loan amount exceeds 3x annual income']


# 3) Prompt Caching

In [6]:
import time

# Simulated cache
prompt_cache = {}

# Simulated LLM call
def call_llm(prompt):
    print("Calling LLM...")

    # Simulate API latency
    time.sleep(2)

    return f"Response for: {prompt}"


def get_response(prompt):

    # Check cache first
    if prompt in prompt_cache:
        print("Retrieved from cache")
        return prompt_cache[prompt]

    # Call LLM
    response = call_llm(prompt)

    # Store in cache
    prompt_cache[prompt] = response

    return response


# -------------------
# Test
# -------------------

prompt1 = "What is Agentic AI?"

start = time.time()
resp = get_response(prompt1)
print(resp)
print("Time:", round(time.time() - start, 2), "seconds")

print("-" * 50)

# Same prompt again
start = time.time()
resp = get_response(prompt1)
print(resp)
print("Time:", round(time.time() - start, 2), "seconds")

print("-" * 50)

# New prompt
start = time.time()
resp = get_response("What is RAG?")
print(resp)
print("Time:", round(time.time() - start, 2), "seconds")

Calling LLM...
Response for: What is Agentic AI?
Time: 2.0 seconds
--------------------------------------------------
Retrieved from cache
Response for: What is Agentic AI?
Time: 0.0 seconds
--------------------------------------------------
Calling LLM...
Response for: What is RAG?
Time: 2.0 seconds


# 4) Agent Evaluation : Test for Accuracy, Relevancy and Completeness

In [9]:
from langchain_openai import ChatOpenAI

In [10]:
llm = ChatOpenAI()

def run(prompt):
    try:
    
        messages = [ ("system", "You are a helpful assistant good in giving information",),
                    ("human", prompt), ]
        
        response = llm.invoke(messages)
        # content = response
        # print(response.content)
    
    except Exception as e:
        response = "Exception from run()." + "\n" + str(e)
    
    return (response)

In [7]:
prompt = ''' 	You are an expert evaluator. Read the Response and evaluate it based on the criteria specified. 
	Provide a structured, objective evaluation without adding unrelated details.
	
    Evaluation Criteria
		Accuracy (0–5): Is the information correct?
		Clarity (0–5): Is the response easy to understand?
		Completeness (0–5): Does it fully address the question?
		Relevance (0–5): Does it stay on-topic?
		Overall Score: Average of the above scores.

	User query:
			"I want to return a pair of shoes I bought last week, but I don’t have the bill. What should I do?"
		
	Response:
			No worries — this happens often. Visit the store where you bought the shoes and explain the situation. Many retailers allow returns without a physical bill as long as they can verify the transaction in their system. If they can’t find the purchase, they might still accept the return at the current selling price or offer an exchange.
'''

In [11]:
response = run(prompt)

In [12]:
print(response.content)

Evaluation:

Accuracy: 5
The information provided is accurate and helpful in guiding the user on how to proceed without a physical bill.

Clarity: 5
The response is clear and easy to understand, providing a straightforward solution to the user's query.

Completeness: 5
The response fully addresses the user's question by explaining the steps to take when returning the shoes without a receipt.

Relevance: 5
The response stays on-topic and provides relevant information related to returning the purchased shoes without a bill.

Overall Score: 5
The response scores high in accuracy, clarity, completeness, and relevance, making it an excellent and helpful guide for the user.


# 5) Asynchronous Process

In [13]:
import time, math, random

In [14]:
# ---------------------
# involves Blocking
# ---------------------

def calc1():
    print("calc1")
    ans = [round(math.sqrt(i),2) for i in range(1,10000)]
    time.sleep(2) # blocks execution
    return(ans)

def calc2():
    print("calc2")
    ans = [ random.randint(1,10000) for i in range(1,10000)]
    time.sleep(2) # blocks execution
    return(ans)

def main():
    start = time.time()
    c1 = calc1()
    c2 = calc2()
    end = time.time() - start

    print(f"Time taken to execute : {round(end,2)} seconds")

main()

calc1
calc2
Time taken to execute : 4.01 seconds


In [15]:
# ---------------------
# Without Blocking
# ---------------------
import asyncio

async def calc3():
    print("calc3")
    ans = [round(math.sqrt(i),2) for i in range(1,10000)]
    await asyncio.sleep(2) # await command is the yield.
    return(ans)

async def calc4():
    print("calc4")
    ans = [random.randint(1, 10000) for i in range(1, 10000)]
    await asyncio.sleep(2)
    return(ans)

async def main():
    start = time.time()
    c1,c2 = await asyncio.gather(calc3(),calc4())
    end = time.time() - start
    print(f"Time taken to execute : {round(end,2)} seconds")

    return(c1,c2)

# execute the functions
# c1,c2 = asyncio.run(main())
c1,c2 = await main()
# c1: stores the return values of func3()
# c2: stores the return values of func4()

print(f"Printing the first 5 values of c1.\n{c1[:5]}")
print()
print(f"Printing the first 5 values of c2.\n{c2[:5]}")

calc3
calc4
Time taken to execute : 2.01 seconds
Printing the first 5 values of c1.
[1.0, 1.41, 1.73, 2.0, 2.24]

Printing the first 5 values of c2.
[2302, 7242, 532, 7073, 1162]


# 6) Security, Safety aspects
# PII : Personal Identifiable Information

In [21]:
import re

In [16]:
def mask_pii(text):
    # Mask email addresses
    text = re.sub(r'\b[\w.-]+@[\w.-]+\.\w+\b', '[EMAIL_MASKED]', text)

    # Mask phone numbers
    text = re.sub(r'\b\d{10}\b', '[PHONE_MASKED]', text)

    # Mask Aadhaar-like numbers: 12 digits
    text = re.sub(r'\b\d{4}\s?\d{4}\s?\d{4}\b', '[AADHAAR_MASKED]', text)

    # Mask PAN-like numbers: ABCDE1234F
    text = re.sub(r'\b[A-Z]{5}[0-9]{4}[A-Z]\b', '[PAN_MASKED]', text)

    return text

In [19]:
# Sample customer message
customer_input = """
Customer name is Ravi Kumar.
Email: ravi.kumar@gmail.com
Phone: 9876543210
Aadhaar: 1234 5678 9012
PAN: ABCDE1234F

Customer wants to apply for a personal loan.
"""

print("Original Text:")
print(customer_input)

Original Text:

Customer name is Ravi Kumar.
Email: ravi.kumar@gmail.com
Phone: 9876543210
Aadhaar: 1234 5678 9012
PAN: ABCDE1234F

Customer wants to apply for a personal loan.



In [22]:
safe_text = mask_pii(customer_input)

print("\nText Sent to LLM:")
print(safe_text)

# AGUI from microsoft


Text Sent to LLM:

Customer name is Ravi Kumar.
Email: [EMAIL_MASKED]
Phone: [PHONE_MASKED]
Aadhaar: [AADHAAR_MASKED]
PAN: [PAN_MASKED]

Customer wants to apply for a personal loan.

